# Giai đoạn 2: Khai phá tập mẫu phổ biến và Luật kết hợp

Mục tiêu của giai đoạn này là triển khai các thuật toán khai phá dữ liệu để tìm ra mối quan hệ giữa các mặt hàng trong tập dữ liệu Groceries. Theo yêu cầu của bài tập lớn, nhóm thực hiện so sánh đánh giá giữa kỹ thuật truyền thống trong môn học và kỹ thuật mới.

* **Nhiệm vụ trọng tâm:** Thực hiện tác vụ khai phá tập mẫu phổ biến và luật kết hợp (L.O.3.4).
* **Thuật toán đối chứng:** Apriori (Trong đề cương - Chương 6) và FP-Growth (Công nghệ mới).

## Bước 1: Thiết lập môi trường và Tải dữ liệu

Trong bước này, chúng ta sử dụng thư viện `mlxtend` để triển khai các thuật toán. Dữ liệu đầu vào là file `basket_matrix.csv` đã được Thành viên 1 xử lý sang dạng ma trận nhị phân (One-hot encoding) ở Giai đoạn 1.

In [47]:
import pandas as pd
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Đọc dữ liệu ma trận giỏ hàng từ Giai đoạn 1
basket_df = pd.read_csv('basket_matrix.csv')

print(f"Dữ liệu sẵn sàng: {basket_df.shape[0]} giao dịch, {basket_df.shape[1]} mặt hàng.")

Dữ liệu sẵn sàng: 14963 giao dịch, 167 mặt hàng.


## Bước 2: Triển khai thuật toán Apriori (Kỹ thuật trong môn học)

Apriori là thuật toán khai phá tập mẫu phổ biến bằng cách lặp lại việc tạo các ứng viên (candidate generation). Đây là nội dung trọng tâm trong Chương 6 của học phần CO3029. Chúng ta sẽ đo lường thời gian thực thi để phục vụ bước so sánh hiệu năng.

In [48]:
# Thiết lập ngưỡng Support tối thiểu (ví dụ 0.2%)
min_support = 0.002 

start_time = time.time()
frequent_itemsets_apriori = apriori(basket_df, min_support=min_support, use_colnames=True)
apriori_time = time.time() - start_time

print(f"Apriori hoàn thành trong: {apriori_time:.4f} giây")
print(f"Số lượng tập phổ biến tìm được: {len(frequent_itemsets_apriori)}")

Apriori hoàn thành trong: 0.2820 giây
Số lượng tập phổ biến tìm được: 330


## Bước 3: Triển khai thuật toán FP-Growth (Kỹ thuật mới)

Để tối ưu hóa quá trình khai phá, nhóm sử dụng thuật toán FP-Growth. Khác với Apriori, FP-Growth sử dụng cấu trúc cây (FP-Tree) để nén dữ liệu và không cần tạo các tập ứng viên trung gian, giúp tăng tốc độ xử lý trên tập dữ liệu lớn.

In [49]:
start_time = time.time()
frequent_itemsets_fpgrowth = fpgrowth(basket_df, min_support=min_support, use_colnames=True)
fpgrowth_time = time.time() - start_time

print(f"FP-Growth hoàn thành trong: {fpgrowth_time:.4f} giây")

FP-Growth hoàn thành trong: 0.1296 giây


In [53]:
import json

# Tập hợp các chỉ số hiệu năng đã đo được
performance_results = {
    'Apriori': round(apriori_time, 4),
    'FP-Growth': round(fpgrowth_time, 4)
}

# Xuất ra file JSON để Thành viên 3 có thể đọc tự động
with open('performance_metrics.json', 'w') as f:
    json.dump(performance_results, f)

print("Đã xuất chỉ số hiệu năng ra file performance_metrics.json thành công!")

Đã xuất chỉ số hiệu năng ra file performance_metrics.json thành công!


## Bước 4: Khai phá Luật kết hợp và Đánh giá

Sau khi tìm được các tập mẫu phổ biến, chúng ta tiến hành tạo các luật kết hợp dựa trên các chỉ số đo lường chất lượng:

* **Support ($s$):** Tần suất xuất hiện của luật trong toàn bộ giao dịch.
* **Confidence ($c$):** Độ tin cậy của luật (Xác suất mua B khi đã biết mua A).
* **Lift ($l$):** Tỷ lệ giữa độ tin cậy của luật và xác suất xuất hiện độc lập của vế phải. Nếu $Lift > 1$, hai mặt hàng có tác động tích cực đến nhau.

In [51]:
# Tạo luật kết hợp với ngưỡng Confidence tối thiểu 10%
rules = association_rules(frequent_itemsets_fpgrowth, metric="lift", min_threshold=1.0)

# Lọc các luật có ý nghĩa (Lift > 1 và Confidence cao)
rules = rules.sort_values(by='lift', ascending=False)

print(f"Đã tạo {len(rules)} luật kết hợp.")
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)



Đã tạo 36 luật kết hợp.


,antecedents,consequents,support,confidence,lift
9,frozenset({curd}),frozenset({sausage}),0.002941,0.087302,1.446615
8,frozenset({sausage}),frozenset({curd}),0.002941,0.048726,1.446615
26,frozenset({brown bread}),frozenset({canned beer}),0.002406,0.063943,1.362937
27,frozenset({canned beer}),frozenset({brown bread}),0.002406,0.051282,1.362937
11,frozenset({frozen vegetables}),frozenset({sausage}),0.002072,0.073986,1.225966
10,frozenset({sausage}),frozenset({frozen vegetables}),0.002072,0.034330,1.225966
21,frozenset({sausage}),frozenset({bottled beer}),0.003342,0.055371,1.222000
20,frozenset({bottled beer}),frozenset({sausage}),0.003342,0.073746,1.222000
6,frozenset({frankfurter}),frozenset({other vegetables}),0.005146,0.136283,1.116150
7,frozenset({other vegetables}),frozenset({frankfurter}),0.005146,0.042146,1.116150


In [52]:
# Chuyển đổi frozenset thành chuỗi để dễ đọc
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Lưu lại file đã làm sạch
rules.to_csv('association_rules_cleaned.csv', index=False)
display(rules.head(10))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
9,curd,sausage,0.033683,0.060349,0.002941,0.087302,1.446615,1.0,0.000908,1.029531,0.319493,0.032282,0.028684,0.068014
8,sausage,curd,0.060349,0.033683,0.002941,0.048726,1.446615,1.0,0.000908,1.015814,0.328559,0.032282,0.015568,0.068014
26,brown bread,canned beer,0.037626,0.046916,0.002406,0.063943,1.362937,1.0,0.000641,1.018191,0.276701,0.029292,0.017866,0.057613
27,canned beer,brown bread,0.046916,0.037626,0.002406,0.051282,1.362937,1.0,0.000641,1.014394,0.279398,0.029292,0.014190,0.057613
11,frozen vegetables,sausage,0.028002,0.060349,0.002072,0.073986,1.225966,1.0,0.000382,1.014726,0.189627,0.024012,0.014513,0.054158
10,sausage,frozen vegetables,0.060349,0.028002,0.002072,0.034330,1.225966,1.0,0.000382,1.006553,0.196155,0.024012,0.006510,0.054158
21,sausage,bottled beer,0.060349,0.045312,0.003342,0.055371,1.222000,1.0,0.000607,1.010649,0.193337,0.032658,0.010537,0.064559
20,bottled beer,sausage,0.045312,0.060349,0.003342,0.073746,1.222000,1.0,0.000607,1.014464,0.190292,0.032658,0.014258,0.064559
6,frankfurter,other vegetables,0.037760,0.122101,0.005146,0.136283,1.116150,1.0,0.000536,1.016420,0.108146,0.033261,0.016154,0.089214
7,other vegetables,frankfurter,0.122101,0.037760,0.005146,0.042146,1.116150,1.0,0.000536,1.004579,0.118536,0.033261,0.004558,0.089214


## Tổng kết Giai đoạn 2

* **Kết quả:** Đã tìm ra 36 luật kết hợp có ý nghĩa thực tế với chỉ số Lift cao nhất đạt 1.446.
* **So sánh:** Thực nghiệm cho thấy Apriori và FP-Growth có sự chênh lệch về thời gian chạy tùy thuộc vào ngưỡng Support tối thiểu.
* **Bàn giao:** Kết quả được xuất ra file `association_rules_results.csv` để Thành viên 3 thực hiện trực quan hóa mạng lưới (Network Graph) ở giai đoạn tiếp theo.